In [0]:
df = spark.read.table("samples.bakehouse.sales_transactions")
display(df)

In [0]:
# from pyspark.sql.functions import col
# df2 = df.where(col("paymentMethod") == "mastercard")
## OR
# df2 = df.where(df['paymentMethod'] == "mastercard")
## OR
df2 = df.where("paymentMethod == 'mastercard'")

display(df2)

In [0]:
%sql
select * from samples.bakehouse.sales_transactions
where paymentMethod == 'mastercard';

In [0]:
from pyspark.sql.functions import *
# df2 = df.withColumn("myCol", pow("quantity",2)/6)
df2 = df.withColumn("myCol", round(pow("quantity",2)/6,2))
display(df2)

In [0]:
%sql
select *, round(pow(quantity,2)/6,2) as newCol
 from samples.bakehouse.sales_transactions;

In [0]:
from pyspark.sql.functions import lower, upper, trim, substr, split, col, lit

df2 = (
    df.select("product")
    .withColumn("lower_product", lower(col("product")))
    .withColumn("upper_product", upper(col("product")))
    .withColumn("trim_product", trim(col("product")))
    .withColumn("substr_product", substr(col("product"), lit(1), lit(4)))
    .withColumn("split_product", split(col("product"), " "))
)

display(df2)

In [0]:
%sql
select product,
lower(product) as lower_product,
upper(product) as upper_product,
trim(product) as trim_product,
substr(product, 1, 4) as substr_product,
split(product, ' ') as split_product
from samples.bakehouse.sales_transactions;

In [0]:
from pyspark.sql.functions import regexp_extract, col

# Extract the middle word using regex: ^\S+\s+(\S+)\s+\S+$
df2 = df.withColumn("middle_word", regexp_extract(col("product"), r"^\S+\s+(\S+)\s+\S+$", 1))

display(df2.select("product", "middle_word"))


##################################################

from pyspark.sql.functions import regexp_replace, col

df2 = df.withColumn(
    "product_replaced",
    regexp_replace(col("product"), r"^Golden Gate Ginger$", "AAAA")
)

display(df2.select("product", "product_replaced"))

In [0]:
%sql
-- Extract the middle word using regex
SELECT
  product,
  regexp_extract(product, '^\S+\s+(\S+)\s+\S+$', 1) AS middle_word
FROM samples.bakehouse.sales_transactions;

In [0]:
%sql
-- Replace 'Golden Gate Ginger' with 'AAAA'
SELECT
  product,
  regexp_replace(product, '^Golden Gate Ginger$', 'AAAA') AS product_replaced
FROM samples.bakehouse.sales_transactions;

In [0]:
from pyspark.sql.functions import current_date, current_timestamp, to_date, date_add, date_sub, lit, to_timestamp

df2 = df.select("dateTime")\
    .withColumn("current_date", current_date())\
        .withColumn("current_timestamp", current_timestamp())\
            .withColumn("date_add", date_add(col("dateTime"), lit(1)))\
                .withColumn("date_sub", date_sub(col("dateTime"), lit(1)))\
                    .withColumn("to_date", to_date(col("dateTime")))\
                        .withColumn("custom_date", to_date(lit("20251014"), 'yyyyMMdd'))\
                            .withColumn("to_timestamp", to_timestamp(col("dateTime"), "MM-dd-yyyy HH:mm:ss"))

display(df2)

In [0]:
%sql
SELECT
  dateTime,
  current_date() AS current_date,
  current_timestamp() AS current_timestamp,
  date_add(dateTime, 1) AS date_add,
  date_sub(dateTime, 1) AS date_sub,
  to_date(dateTime) AS to_date,
  to_date('20251014', 'yyyyMMdd') AS custom_date,
  to_timestamp(dateTime, 'MM-dd-yyyy HH:mm:ss') AS to_timestamp
FROM samples.bakehouse.sales_transactions;

In [0]:
from pyspark.sql.functions import coalesce

df2 = df.select("transactionId", "customerID")\
    .withColumn("coalesce", coalesce(col("transactionId"), col("customerID")))\
        .withColumn("if_else", expr("CASE WHEN transactionId IS NOT NULL THEN transactionId ELSE customerID END"))\
            .na.drop(subset = ["transactionId"])

# .na.drop('all')
# .na.fill('myvalue')

display(df2)

In [0]:
%sql
SELECT
  transactionId,
  customerID,
  COALESCE(transactionId, customerID) AS coalesce,
  CASE WHEN transactionId IS NOT NULL THEN transactionId ELSE customerID END AS if_else
FROM samples.bakehouse.sales_transactions;